In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn"]
ASSETS = [{"name": "auto-mpg.csv", "url": "/labs/2.1.1/auto-mpg.csv", "size": 17709, "sha256": "edf98579b55f319d705ff9a6643f28191ed4bafa997049e271ee8c271fdfa38a"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.1.1 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd

# 加载数据集并显示数据集的前五行 1分
data = __________
print("数据集的前五行:")
print(__________)

# 显示每一列的数据类型
print(data.dtypes)

# 检查缺失值并删除缺失值所在的行  2分
print("\n检查缺失值:")
print(__________.__________.__________)  
data = __________

# 将 'horsepower' 列转换为数值类型，并（删除）处理转换中的异常值 1分
data['horsepower'] = __________(data['horsepower'], errors='coerce')
data = __________

# 显示每一列的数据类型
print(data.horsepower.dtypes)

# 检查清洗后的缺失值
print("\n检查清洗后的缺失值:")
print(data.isnull().sum())

from sklearn.preprocessing import StandardScaler
# 对数值型数据进行标准化处理 1分
numerical_features = ['displacement', 'horsepower', 'weight', 'acceleration']
scaler = StandardScaler()
data[numerical_features] = __________

from sklearn.model_selection import train_test_split
# 选择特征、自变量和目标变量 2分
selected_features = __________
X = __________
y = __________

# 划分数据集为训练集和测试集（训练集占8成） 1分
X_train, X_test, y_train, y_test = __________(__________, random_state=42)


# 将特征和目标变量合并到一个数据框中
cleaned_data = X.copy()
cleaned_data['mpg'] = y

# 保存清洗和处理后的数据（不存储额外的索引号） 1分
__________('2.1.1_cleaned_data.csv', __________)

# 打印消息指示文件已保存
print("\n清洗后的数据已保存到 2.1.1_cleaned_data.csv")